<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/secondWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 - makemore part 2 (MLP) + part 3 (BatchNorm & internals)

First week was micrograd. This week I move ahead with makemore -> first a proper MLP (the Bengio 2003 one), then its internals, weight init and batch norm.

I have solved the assignment exercises inside this same notebook (Part 1 -> E01, E02, E03 and Part 2 -> E01, E02).

In [ ]:
# download karpathy's names.txt....
import urllib.request
import math
import torch
import torch.nn.functional as F

url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')
words = open('names.txt').read().splitlines()

# vocab.... char to index and back
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('names:', len(words), '| vocab:', vocab_size)

# Part 1 - MLP (makemore part 2)

block_size = 3 previous characters as context. every character is embedded into a small vector (lookup table C). split the dataset into 80/10/10.

In [ ]:
block_size = 3


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size          # start with all '.'
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]     # sliding window
    return torch.tensor(X), torch.tensor(Y)


import random
random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print('train:', Xtr.shape, '| dev:', Xdev.shape, '| test:', Xte.shape)

In [ ]:
# small helpers so I don't rewrite the same loop again and again
def make_params(n_embd=10, n_hidden=200, seed=2147483647, last_scale=0.01, kaiming=True):
    g = torch.Generator().manual_seed(seed)
    fan_in = n_embd * block_size
    C = torch.randn((vocab_size, n_embd), generator=g)
    # kaiming init -> tanh gain 5/3 divided by sqrt(fan_in)
    W1 = torch.randn((fan_in, n_hidden), generator=g)
    W1 = W1 * ((5 / 3) / fan_in ** 0.5) if kaiming else W1
    b1 = torch.randn(n_hidden, generator=g) * 0.01
    W2 = torch.randn((n_hidden, vocab_size), generator=g) * last_scale  # keep it small
    b2 = torch.randn(vocab_size, generator=g) * 0
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True
    return params


def forward_mlp(params, Xb):
    C, W1, b1, W2, b2 = params
    emb = C[Xb]                              # embedding lookup here
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1 + b1)
    logits = h @ W2 + b2
    return logits


def train_mlp(params, steps=20000, batch_size=32):
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,))
        loss = F.cross_entropy(forward_mlp(params, Xtr[ix]), Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        lr = 0.1 if i < 0.6 * steps else 0.01   # decay lr later
        for p in params:
            p.data += -lr * p.grad
        if i % 4000 == 0:
            print(f'{i:6d}: {loss.item():.4f}')


@torch.no_grad()
def eval_mlp(params, split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    loss = F.cross_entropy(forward_mlp(params, X), Y)
    print(split, 'loss:', loss.item())
    return loss.item()

## E01 -> tune the hyperparameters to beat Andrej's best validation loss of 2.2

n_embd=10, n_hidden=200, block_size=3, 20k steps and an lr schedule (0.1 -> 0.01). this brings the dev loss below 2.2.

In [ ]:
params = make_params(n_embd=10, n_hidden=200)
print('total params:', sum(p.nelement() for p in params))
train_mlp(params, steps=20000)

eval_mlp(params, 'train')
dev = eval_mlp(params, 'dev')
print('\nAndrej best was 2.2 -> our dev', round(dev, 4), '(beaten)' if dev < 2.2 else '(still above)')

## E02 -> Initialization matters

question: what loss would we get if the predicted probabilities at init were perfectly uniform? and what do we actually get?

uniform loss = -log(1/27) = 3.2958.... (27 characters with equal probability). with a good init the starting loss should be around this. if the last layer is initialized large, the loss is very high (confidently wrong).

In [ ]:
uniform_loss = -math.log(1 / 27)
print('uniform loss (ideal starting):', round(uniform_loss, 4))

# bad init -> last layer large (last_scale=1) and W1 without kaiming
bad = make_params(last_scale=1.0, kaiming=False)
with torch.no_grad():
    bad_start = F.cross_entropy(forward_mlp(bad, Xtr[:1000]), Ytr[:1000]).item()
print('bad init starting loss :', round(bad_start, 4))

# good init -> last layer small (last_scale=0.01)
good = make_params(last_scale=0.01)
with torch.no_grad():
    good_start = F.cross_entropy(forward_mlp(good, Xtr[:1000]), Ytr[:1000]).item()
print('good init starting loss:', round(good_start, 4), '  <- very close to uniform')

so making W2 small (*0.01) and b2 = 0 brought the starting loss to ~3.29 (like uniform), while the bad init started at 20+. this way the first few thousand steps are not wasted just bringing the loss down.

## E03 -> implement an idea from the Bengio et al. 2003 paper

the paper has an optional part -> **direct connections**, i.e. a linear path straight from the input (embedding concat) to the output (`embcat @ W3`). I tried that and compared it against the plain MLP.

In [ ]:
# model with a direct connection: logits = h @ W2 + b2 + embcat @ W3
g = torch.Generator().manual_seed(2147483647)
n_embd, n_hidden = 10, 200
fan_in = n_embd * block_size
Cd = torch.randn((vocab_size, n_embd), generator=g)
W1d = torch.randn((fan_in, n_hidden), generator=g) * (5 / 3) / fan_in ** 0.5
b1d = torch.randn(n_hidden, generator=g) * 0.01
W2d = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2d = torch.randn(vocab_size, generator=g) * 0
W3d = torch.randn((fan_in, vocab_size), generator=g) * 0.01     # direct path
paramsd = [Cd, W1d, b1d, W2d, b2d, W3d]
for p in paramsd:
    p.requires_grad = True

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = Cd[Xtr[ix]]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1d + b1d)
    logits = h @ W2d + b2d + embcat @ W3d      # <- direct connection added
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in paramsd:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 12000 else 0.01
    for p in paramsd:
        p.data += -lr * p.grad


@torch.no_grad()
def eval_direct(X, Y):
    emb = Cd[X]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1d + b1d)
    logits = h @ W2d + b2d + embcat @ W3d
    return F.cross_entropy(logits, Y).item()


print('direct-connection dev loss:', round(eval_direct(Xdev, Ydev), 4))
# my observation -> barely any difference here (dataset is simple), but the paper says
# on larger vocabularies this helps training go faster.

# Part 2 - Internals & BatchNorm (makemore part 3)

now add batch normalization to the MLP. in every batch we normalize the hidden pre-activation (mean 0, std 1), then apply learnable gamma (bngain) and beta (bnbias). we also keep a running mean/std for inference. this makes training stable and less sensitive to the exact init.

In [ ]:
n_embd, n_hidden = 10, 200
g = torch.Generator().manual_seed(2147483647)
fan_in = n_embd * block_size

C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((fan_in, n_hidden), generator=g) * (5 / 3) / fan_in ** 0.5
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0
# batchnorm parameters (no b1 needed since bn subtracts the mean anyway)
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

bn_params = [C, W1, W2, b2, bngain, bnbias]
for p in bn_params:
    p.requires_grad = True

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1
    # ---- batch norm ----
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
    # --------------------
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in bn_params:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 12000 else 0.01
    for p in bn_params:
        p.data += -lr * p.grad
    if i % 4000 == 0:
        print(f'{i:6d}: {loss.item():.4f}')

In [ ]:
# dev loss (using the running stats)
@torch.no_grad()
def eval_bn(X, Y):
    emb = C[X]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, Y).item()


print('batchnorm dev loss:', round(eval_bn(Xdev, Ydev), 4))

## Part 2 E01 -> initialize all weights and biases to zero and train

one of three things could happen -> (1) the net trains fine (2) it doesn't train at all (3) it trains but only partially with a bad final result. answer = (3).

I ran one forward + backward and looked at the gradients.... with everything zero, the gradients of C, W1, b1, W2 come out zero, and only **b2** (the output bias) has a non-zero gradient.

In [ ]:
# plain MLP (no batchnorm) with all params zero
zC = torch.zeros((vocab_size, n_embd), requires_grad=True)
zW1 = torch.zeros((n_embd * block_size, n_hidden), requires_grad=True)
zb1 = torch.zeros(n_hidden, requires_grad=True)
zW2 = torch.zeros((n_hidden, vocab_size), requires_grad=True)
zb2 = torch.zeros(vocab_size, requires_grad=True)
zparams = [zC, zW1, zb1, zW2, zb2]
names = ['C', 'W1', 'b1', 'W2', 'b2']


def zforward(Xb):
    emb = zC[Xb]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ zW1 + zb1)
    return h @ zW2 + zb2


ix = torch.randint(0, Xtr.shape[0], (32,))
loss = F.cross_entropy(zforward(Xtr[ix]), Ytr[ix])
for p in zparams:
    p.grad = None
loss.backward()

print('starting loss:', round(loss.item(), 4), '(equal to uniform)')
for nm, p in zip(names, zparams):
    print(f'  {nm:3s} grad norm: {p.grad.norm().item():.6f}')

In [ ]:
# now train it -> the loss drops a bit and then plateaus
for i in range(5000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    loss = F.cross_entropy(zforward(Xtr[ix]), Ytr[ix])
    for p in zparams:
        p.grad = None
    loss.backward()
    for p in zparams:
        p.data += -0.1 * p.grad
    if i % 1000 == 0:
        print(f'{i:5d}: {loss.item():.4f}')

print('\nnotice C/W1/b1/W2 are still zero:')
for nm, p in zip(names, zparams):
    print(f'  {nm:3s} sum(|.|): {p.detach().abs().sum().item():.4f}')

**what is happening (my understanding):** everything is zero, so `h = tanh(0) = 0`, and the logits depend only on `b2`.

- `dW2 = h.T @ dlogits` -> h is zero, so this is zero
- `dh = dlogits @ W2.T` -> W2 is zero, so this is zero -> which makes the gradients of W1, b1, C zero too
- only `db2` is non-zero

so the network can only learn the **unigram** distribution (how often each letter appears) through `b2`, while the rest of the net stays stuck at zero -> that is the 'partial training' and the bad final loss. lesson -> you need a random init to break the symmetry.

## Part 2 E02 -> fold BatchNorm into the preceding Linear

after training, the batchnorm gamma/beta can be folded into the previous Linear layer's W,b, so at inference time we don't need the batchnorm at all (faster test time).

math: at inference bn does `y = gamma*(x - mean)/sqrt(var+eps) + beta`, and `x = W@inp + b`. both are affine, so they combine into ->
`W_new = W * scale`, `b_new = beta + (b - mean)*scale`, where `scale = gamma/sqrt(var+eps)`.

I built a 3-layer MLP (a batchnorm after each linear), trained it, then folded and verified the forward pass is the same.

In [ ]:
# small module classes (for the fold demo)
class Linear:
    def __init__(self, fan_in, fan_out):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out)

    def __call__(self, x):
        return x @ self.weight + self.bias


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps, self.momentum, self.training = eps, momentum, True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        else:
            xmean, xvar = self.running_mean, self.running_var
        return self.gamma * (x - xmean) / torch.sqrt(xvar + self.eps) + self.beta


torch.manual_seed(1)
# 3 linear + 3 batchnorm (input = embcat of size n_embd*block_size = 30)
L1, B1 = Linear(30, 100), BatchNorm1d(100)
L2, B2 = Linear(100, 100), BatchNorm1d(100)
L3, B3 = Linear(100, vocab_size), BatchNorm1d(vocab_size)
layers = [L1, B1, L2, B2, L3, B3]
Cf = torch.randn((vocab_size, n_embd))

params = [Cf] + [L1.weight, L1.bias, B1.gamma, B1.beta,
                 L2.weight, L2.bias, B2.gamma, B2.beta,
                 L3.weight, L3.bias, B3.gamma, B3.beta]
for p in params:
    p.requires_grad = True


def forward3(Xb):
    x = Cf[Xb].view(Xb.shape[0], -1)
    x = torch.tanh(B1(L1(x)))
    x = torch.tanh(B2(L2(x)))
    x = B3(L3(x))
    return x


for i in range(3000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    loss = F.cross_entropy(forward3(Xtr[ix]), Ytr[ix])
    for p in params:
        p.grad = None
    loss.backward()
    for p in params:
        p.data += -0.1 * p.grad
print('trained 3-layer bn net, last loss:', round(loss.item(), 4))

In [ ]:
# now fold and verify
for L in layers:
    if isinstance(L, BatchNorm1d):
        L.training = False       # inference mode -> running stats

Xb = Xdev[:200]
with torch.no_grad():
    out_original = forward3(Xb)


def fold(linear, bn):
    """merge the batchnorm that follows this linear into the linear itself."""
    scale = bn.gamma / torch.sqrt(bn.running_var + bn.eps)   # per-output feature
    new = Linear(linear.weight.shape[0], linear.weight.shape[1])
    with torch.no_grad():
        new.weight = linear.weight * scale
        new.bias = bn.beta + (linear.bias - bn.running_mean.squeeze()) * scale.squeeze()
    return new


F1 = fold(L1, B1)
F2 = fold(L2, B2)
F3 = fold(L3, B3)


def forward_folded(Xb):
    # no batchnorm.... just the folded linear layers
    x = Cf[Xb].view(Xb.shape[0], -1)
    x = torch.tanh(F1(x))
    x = torch.tanh(F2(x))
    x = F3(x)
    return x


with torch.no_grad():
    out_folded = forward_folded(Xb)

maxdiff = (out_original - out_folded).abs().max().item()
print('max difference between original and folded:', maxdiff)
print('same forward pass?', torch.allclose(out_original, out_folded, atol=1e-5))

# Week 2 summary (my notes)

- MLP + character embeddings (makemore part 2) got the dev loss below 2.2 (E01)
- init matters -> uniform loss is 3.29, and shrinking the last layer made the starting loss land right there (E02)
- tried bengio's direct-connection idea -> very small difference on this small dataset (E03)
- batchnorm (makemore part 3) made training stable
- all-zero init -> only b2 trains, so the net only learns the unigram (Part 2 E01)
- folded batchnorm into the linear and verified the same inference output (Part 2 E02)

next week -> makemore part 4 (backprop ninja) and part 5 (wavenet)....

<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/secondWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Week 2 = makemore part 2 (MLP) + part 3 (BatchNorm & internals)**

first week micrograd tha. is week hum makemore ko aage badhate hai -> pehle ek proper MLP (Bengio 2003 wala), phir uske internals, weight init aur batch norm.

assignment ke exercises bhi isi notebook me solve kiye hai (Part 1 -> E01, E02, E03 aur Part 2 -> E01, E02).

## main resources
- makemore part 2 (MLP): https://youtu.be/TCH_1BHY58I
- makemore part 3 (batchnorm, init, internals): https://youtu.be/P6sfmUTpUmc

## other resources (papers.... inhe padhna zaroori hai)
- batch norm paper (Ioffe & Szegedy 2015): https://arxiv.org/abs/1502.03167
- cs231n notes (dropout / batchnorm / init): https://cs231n.github.io/neural-networks-2/
- kaiming init paper (He et al. 2015): https://arxiv.org/abs/1502.01852
- init nicely explained: https://pouannes.github.io/blog/initialization/
- backprop through batchnorm: https://kratzert.github.io/2016/02/12/understanding-the-gradient-flow-through-the-batch-normalization-layer.html

In [ ]:
# karpathy ka names.txt download karte hai....
import urllib.request
import math
import torch
import torch.nn.functional as F

url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')
words = open('names.txt').read().splitlines()

# vocab.... char to index and back
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('names:', len(words), '| vocab:', vocab_size)

# **Part 1 = MLP (makemore part 2)**

block_size = 3 previous chars ka context. har char ko ek chhoti vector me embed karte hai (lookup table C). dataset ko 80/10/10 me todte hai.

In [ ]:
block_size = 3


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size          # start me sab '.'
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]     # sliding window
    return torch.tensor(X), torch.tensor(Y)


import random
random.seed(42)
random.shuffle(words)
n1, n2 = int(0.8 * len(words)), int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print('train:', Xtr.shape, '| dev:', Xdev.shape, '| test:', Xte.shape)

In [ ]:
# chhote helper bana leta hu taaki baar baar wahi loop na likhna pade
def make_params(n_embd=10, n_hidden=200, seed=2147483647, last_scale=0.01, kaiming=True):
    g = torch.Generator().manual_seed(seed)
    fan_in = n_embd * block_size
    C = torch.randn((vocab_size, n_embd), generator=g)
    # kaiming init -> tanh gain 5/3 divided by sqrt(fan_in)
    W1 = torch.randn((fan_in, n_hidden), generator=g)
    W1 = W1 * ((5 / 3) / fan_in ** 0.5) if kaiming else W1
    b1 = torch.randn(n_hidden, generator=g) * 0.01
    W2 = torch.randn((n_hidden, vocab_size), generator=g) * last_scale  # chhota rakho
    b2 = torch.randn(vocab_size, generator=g) * 0
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True
    return params


def forward_mlp(params, Xb):
    C, W1, b1, W2, b2 = params
    emb = C[Xb]                              # yaha embedding lookup
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1 + b1)
    logits = h @ W2 + b2
    return logits


def train_mlp(params, steps=20000, batch_size=32):
    for i in range(steps):
        ix = torch.randint(0, Xtr.shape[0], (batch_size,))
        loss = F.cross_entropy(forward_mlp(params, Xtr[ix]), Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        lr = 0.1 if i < 0.6 * steps else 0.01   # baad me lr kam
        for p in params:
            p.data += -lr * p.grad
        if i % 4000 == 0:
            print(f'{i:6d}: {loss.item():.4f}')


@torch.no_grad()
def eval_mlp(params, split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    loss = F.cross_entropy(forward_mlp(params, X), Y)
    print(split, 'loss:', loss.item())
    return loss.item()

## E01 -> hyperparameters tune karke Andrej ka best val loss 2.2 beat karna

n_embd=10, n_hidden=200, block_size=3, 20k steps aur lr schedule (0.1 -> 0.01). is se dev loss 2.2 se neeche aa jata hai.

In [ ]:
params = make_params(n_embd=10, n_hidden=200)
print('total params:', sum(p.nelement() for p in params))
train_mlp(params, steps=20000)

eval_mlp(params, 'train')
dev = eval_mlp(params, 'dev')
print('\nAndrej ka best 2.2 tha -> humara dev', round(dev, 4), '(beat kiya)' if dev < 2.2 else '(abhi upar hai)')

## E02 -> Initialization matters

sawaal: agar shuruaat me saari predicted probabilities perfectly uniform hoti to loss kya aata? aur hum actually kya paate hai?

uniform loss = -log(1/27) = 3.2958.... (kyunki 27 characters pe barabar probability). agar init acha ho to starting loss isi ke aas paas hona chahiye. agar last layer bada init karo to loss bahut high (confidently galat).

In [ ]:
uniform_loss = -math.log(1 / 27)
print('uniform loss (ideal starting):', round(uniform_loss, 4))

# bad init -> last layer bada (last_scale=1) aur W1 bhi bina kaiming
bad = make_params(last_scale=1.0, kaiming=False)
with torch.no_grad():
    bad_start = F.cross_entropy(forward_mlp(bad, Xtr[:1000]), Ytr[:1000]).item()
print('bad init starting loss :', round(bad_start, 4))

# good init -> last layer chhota (last_scale=0.01)
good = make_params(last_scale=0.01)
with torch.no_grad():
    good_start = F.cross_entropy(forward_mlp(good, Xtr[:1000]), Ytr[:1000]).item()
print('good init starting loss:', round(good_start, 4), '  <- uniform ke bahut kareeb')

matlab W2 ko chhota (*0.01) aur b2 ko 0 rakhne se starting loss ~3.29 aa gaya (uniform jaisa), jabki bad init me ye 20+ tha. is se pehle ke kai hazaar steps 'waste' nahi hote sirf loss ko neeche laane me.

## E03 -> Bengio et al. 2003 paper ka koi idea implement karna

paper me ek optional cheez hai -> **direct connections** yaani input (embedding concat) se seedha output tak ek linear path (`embcat @ W3`). maine wahi try kiya aur normal MLP se compare kiya.

In [ ]:
# direct connection wala model: logits = h @ W2 + b2 + embcat @ W3
g = torch.Generator().manual_seed(2147483647)
n_embd, n_hidden = 10, 200
fan_in = n_embd * block_size
Cd = torch.randn((vocab_size, n_embd), generator=g)
W1d = torch.randn((fan_in, n_hidden), generator=g) * (5 / 3) / fan_in ** 0.5
b1d = torch.randn(n_hidden, generator=g) * 0.01
W2d = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2d = torch.randn(vocab_size, generator=g) * 0
W3d = torch.randn((fan_in, vocab_size), generator=g) * 0.01     # direct path
paramsd = [Cd, W1d, b1d, W2d, b2d, W3d]
for p in paramsd:
    p.requires_grad = True

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = Cd[Xtr[ix]]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1d + b1d)
    logits = h @ W2d + b2d + embcat @ W3d      # <- direct connection add
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in paramsd:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 12000 else 0.01
    for p in paramsd:
        p.data += -lr * p.grad


@torch.no_grad()
def eval_direct(X, Y):
    emb = Cd[X]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ W1d + b1d)
    logits = h @ W2d + b2d + embcat @ W3d
    return F.cross_entropy(logits, Y).item()


print('direct-connection dev loss:', round(eval_direct(Xdev, Ydev), 4))
# mera observation -> yaha bahut chhota hi farak pada (dataset simple hai), but paper me
# bade vocab pe ye faster training me help karta hai.

# **Part 2 = Internals & BatchNorm (makemore part 3)**

ab MLP me batch normalization add karte hai. har batch me hidden pre-activation ko normalize (mean 0, std 1) kar dete hai, phir seekhne wale gamma (bngain) aur beta (bnbias) laga dete hai. running mean/std bhi rakhte hai inference ke liye. is se training stable ho jati hai aur init pe itna dependent nahi rehte.

In [ ]:
n_embd, n_hidden = 10, 200
g = torch.Generator().manual_seed(2147483647)
fan_in = n_embd * block_size

C = torch.randn((vocab_size, n_embd), generator=g)
W1 = torch.randn((fan_in, n_hidden), generator=g) * (5 / 3) / fan_in ** 0.5
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0
# batchnorm ke parameters (b1 ki zaroorat nahi kyunki bn khud mean hata deta hai)
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

bn_params = [C, W1, W2, b2, bngain, bnbias]
for p in bn_params:
    p.requires_grad = True

for i in range(20000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    emb = C[Xtr[ix]]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1
    # ---- batch norm ----
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi
    # --------------------
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Ytr[ix])
    for p in bn_params:
        p.grad = None
    loss.backward()
    lr = 0.1 if i < 12000 else 0.01
    for p in bn_params:
        p.data += -lr * p.grad
    if i % 4000 == 0:
        print(f'{i:6d}: {loss.item():.4f}')

In [ ]:
# dev loss (running stats use karke)
@torch.no_grad()
def eval_bn(X, Y):
    emb = C[X]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, Y).item()


print('batchnorm dev loss:', round(eval_bn(Xdev, Ydev), 4))

## Part 2 E01 -> saare weights aur biases zero se init karo

expectation 3 me se ek -> (1) net theek train hoga (2) bilkul nahi hoga (3) partial train hoga aur bura final result. answer = (3).

maine ek forward + backward chala ke gradients dekhe.... sab zero hone pe C, W1, b1, W2 ke gradients zero aate hai, sirf **b2** (output bias) ka gradient non-zero aata hai.

In [ ]:
# plain MLP (no batchnorm) with all params zero
zC = torch.zeros((vocab_size, n_embd), requires_grad=True)
zW1 = torch.zeros((n_embd * block_size, n_hidden), requires_grad=True)
zb1 = torch.zeros(n_hidden, requires_grad=True)
zW2 = torch.zeros((n_hidden, vocab_size), requires_grad=True)
zb2 = torch.zeros(vocab_size, requires_grad=True)
zparams = [zC, zW1, zb1, zW2, zb2]
names = ['C', 'W1', 'b1', 'W2', 'b2']


def zforward(Xb):
    emb = zC[Xb]
    embcat = emb.view(emb.shape[0], -1)
    h = torch.tanh(embcat @ zW1 + zb1)
    return h @ zW2 + zb2


ix = torch.randint(0, Xtr.shape[0], (32,))
loss = F.cross_entropy(zforward(Xtr[ix]), Ytr[ix])
for p in zparams:
    p.grad = None
loss.backward()

print('starting loss:', round(loss.item(), 4), '(uniform ke barabar)')
for nm, p in zip(names, zparams):
    print(f'  {nm:3s} grad norm: {p.grad.norm().item():.6f}')

In [ ]:
# ab isko train karo -> loss thoda girega phir plateau ho jayega
for i in range(5000):
    ix = torch.randint(0, Xtr.shape[0], (32,))
    loss = F.cross_entropy(zforward(Xtr[ix]), Ytr[ix])
    for p in zparams:
        p.grad = None
    loss.backward()
    for p in zparams:
        p.data += -0.1 * p.grad
    if i % 1000 == 0:
        print(f'{i:5d}: {loss.item():.4f}')

print('\ndekho C/W1/b1/W2 abhi bhi zero hai:')
for nm, p in zip(names, zparams):
    print(f'  {nm:3s} sum(|.|): {p.detach().abs().sum().item():.4f}')

**kya ho raha hai (mera samajh):** sab zero hai to `h = tanh(0) = 0`, aur logits sirf `b2` pe depend karte hai.

- `dW2 = h.T @ dlogits` -> h zero hai to ye zero
- `dh = dlogits @ W2.T` -> W2 zero hai to ye zero -> isse W1, b1, C sab ke gradients zero
- sirf `db2` non-zero

to network sirf `b2` ke through **unigram** (kaunsa letter kitni baar aata hai) seekh pata hai, baaki poora net zero pe stuck reh jata hai -> isliye 'partial training' aur bura final loss. lesson -> symmetry todne ke liye random init chahiye.

## Part 2 E02 -> BatchNorm ko preceding Linear me 'fold' karna

training ke baad batchnorm ke gamma/beta ko pichle Linear layer ke W,b me mila (fold) sakte hai, aur inference ke waqt batchnorm ki zaroorat hi nahi padegi (test time faster).

math: bn inference me `y = gamma*(x - mean)/sqrt(var+eps) + beta`, aur `x = W@inp + b`. dono affine hai to combine kar sakte hai ->
`W_new = W * scale`, `b_new = beta + (b - mean)*scale`, jaha `scale = gamma/sqrt(var+eps)`.

maine 3-layer MLP (har linear ke baad batchnorm) banaya, train kiya, phir fold karke verify kiya ki forward same aata hai.

In [ ]:
# chhote module classes (fold demo ke liye)
class Linear:
    def __init__(self, fan_in, fan_out):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out)

    def __call__(self, x):
        return x @ self.weight + self.bias


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps, self.momentum, self.training = eps, momentum, True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(0, keepdim=True)
            xvar = x.var(0, keepdim=True)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar
        else:
            xmean, xvar = self.running_mean, self.running_var
        return self.gamma * (x - xmean) / torch.sqrt(xvar + self.eps) + self.beta


torch.manual_seed(1)
# 3 linear + 3 batchnorm (input = embcat of size n_embd*block_size = 30)
L1, B1 = Linear(30, 100), BatchNorm1d(100)
L2, B2 = Linear(100, 100), BatchNorm1d(100)
L3, B3 = Linear(100, vocab_size), BatchNorm1d(vocab_size)
layers = [L1, B1, L2, B2, L3, B3]
Cf = torch.randn((vocab_size, n_embd))

params = [Cf] + [L1.weight, L1.bias, B1.gamma, B1.beta,
                 L2.weight, L2.bias, B2.gamma, B2.beta,
                 L3.weight, L3.bias, B3.gamma, B3.beta]
for p in params:
    p.requires_grad = True


def forward3(Xb):
    x = Cf[Xb].view(Xb.shape[0], -1)
    x = torch.tanh(B1(L1(x)))
    x = torch.tanh(B2(L2(x)))
    x = B3(L3(x))
    return x


for i in range(3000):
    ix = torch.randint(0, Xtr.shape[0], (64,))
    loss = F.cross_entropy(forward3(Xtr[ix]), Ytr[ix])
    for p in params:
        p.grad = None
    loss.backward()
    for p in params:
        p.data += -0.1 * p.grad
print('trained 3-layer bn net, last loss:', round(loss.item(), 4))

In [ ]:
# ab fold karo aur verify karo
for L in layers:
    if isinstance(L, BatchNorm1d):
        L.training = False       # inference mode -> running stats

Xb = Xdev[:200]
with torch.no_grad():
    out_original = forward3(Xb)


def fold(linear, bn):
    """linear ke baad wale batchnorm ko usi linear me merge kar do."""
    scale = bn.gamma / torch.sqrt(bn.running_var + bn.eps)   # per-output feature
    new = Linear(linear.weight.shape[0], linear.weight.shape[1])
    with torch.no_grad():
        new.weight = linear.weight * scale
        new.bias = bn.beta + (linear.bias - bn.running_mean.squeeze()) * scale.squeeze()
    return new


F1 = fold(L1, B1)
F2 = fold(L2, B2)
F3 = fold(L3, B3)


def forward_folded(Xb):
    # koi batchnorm nahi.... sirf folded linear layers
    x = Cf[Xb].view(Xb.shape[0], -1)
    x = torch.tanh(F1(x))
    x = torch.tanh(F2(x))
    x = F3(x)
    return x


with torch.no_grad():
    out_folded = forward_folded(Xb)

maxdiff = (out_original - out_folded).abs().max().item()
print('max difference between original and folded:', maxdiff)
print('same forward pass?', torch.allclose(out_original, out_folded, atol=1e-5))

# **week 2 ka summary (mera notes)**

- MLP + character embeddings (makemore part 2) se dev loss 2.2 se neeche laa paye (E01)
- init matters -> uniform loss 3.29 hai, last layer chhota karke starting loss wahi ke paas laaya (E02)
- bengio ka direct-connection idea try kiya -> is chhote dataset pe farak bahut kam (E03)
- batchnorm (makemore part 3) se training stable
- sab zero init -> sirf b2 train hota hai, unigram tak seekhta hai (Part2 E01)
- batchnorm ko linear me fold karke inference ka same output verify kiya (Part2 E02)

next week -> makemore part 4 (backprop ninja) aur part 5 (wavenet)....

<a href="https://colab.research.google.com/github/Nitesh-9009/TokensToTranslation/blob/main/secondWeek_SOC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **makemore = character level language model**

first week me micrograd banaya tha.... ab hum karpathy ke makemore ko follow karenge.

step by step chalenge -> bigram -> bigram as neural net -> mlp with character embeddings -> weight init -> batch norm.

dataset = 32k naam (names.txt).

In [ ]:
# first we download karpathy ka names.txt....
import urllib.request

url = 'https://raw.githubusercontent.com/karpathy/makemore/master/names.txt'
urllib.request.urlretrieve(url, 'names.txt')

words = open('names.txt').read().splitlines()
print('total names:', len(words))
print(words[:8])

# **bigram model = just counting**

idea bahut simple hai.... har character ke baad kaunsa character aata hai bas uski ginti kar lo. '.' start aur end ka special token hai.

In [ ]:
import torch

# now we make the vocabulary.... char to index and index to char
chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0                       # '.' ko index 0 diya
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)
print('vocab size:', vocab_size)

# 27x27 count matrix.... which char comes after which
N = torch.zeros((27, 27), dtype=torch.int32)
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[stoi[ch1], stoi[ch2]] += 1

In [ ]:
# now we convert the counts into probability.... +1 is smoothing so no zero prob
P = (N + 1).float()
P /= P.sum(1, keepdim=True)          # har row ka sum 1 ho jaye

# ab hum is bigram model se kuch naam sample karte hai
g = torch.Generator().manual_seed(2147483647)
for _ in range(5):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

In [ ]:
# how good is the model.... we check using negative log likelihood (lower is better)
log_likelihood = 0.0
n = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]
        log_likelihood += torch.log(prob)
        n += 1
print('nll:', (-log_likelihood / n).item())

# **same bigram but as a neural net**

ab wahi bigram hum ek neural net se seekhenge.... input char ko one-hot banao, ek linear layer W se multiply, phir softmax. end me loss almost same aata hai (jaisa hona chahiye).

In [ ]:
import torch.nn.functional as F

# training set banate hai.... x = current char, y = next char
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples:', num)

g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# gradient descent loop.... same idea as micrograd -> forward, backward, update
for k in range(120):
    xenc = F.one_hot(xs, num_classes=27).float()
    logits = xenc @ W                       # log-counts jaisa
    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)   # softmax
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()

    W.grad = None
    loss.backward()
    W.data += -50 * W.grad

print('final loss:', loss.item())
# ye almost bigram counts jitna hi aata hai.... matlab net ne wahi seekh liya

# **MLP model = Bengio 2003 paper**

bigram me hum sirf 1 previous char dekhte the.

Ab context badhate hai -> block_size = 3 previous chars.

Aur har char ko ek chhoti vector me convert karenge -> yahi character embedding hai (lookup table C).

dataset ko 80/10/10 me todte hai (train / dev / test).

In [ ]:
block_size = 3      # how many previous chars we look at


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size          # start me sab '.'
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]     # sliding window aage khiskao
    return torch.tensor(X), torch.tensor(Y)


import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print('train:', Xtr.shape, 'dev:', Xdev.shape, 'test:', Xte.shape)

# **weight init + batch normalization**

do important cheez jo makemore part 3 me sikhaya:

1. weight init -> agar weights bade random se start ho to starting loss bahut high aata hai aur tanh saturate (dead) ho jata hai. isliye W1 ko kaiming init (5/3 / sqrt(fan_in)) aur W2 ko chhota (*0.01) rakhte hai.

2. batch norm -> hidden pre-activation ko har batch me normalize kar dete hai (mean 0, std 1).... training smooth ho jati hai. running mean/std bhi rakhte hai inference ke time ke liye.

In [ ]:
n_embd = 10        # embedding size per char
n_hidden = 200     # hidden layer size
g = torch.Generator().manual_seed(2147483647)

C = torch.randn((vocab_size, n_embd), generator=g)
# kaiming init.... tanh gain 5/3 divided by sqrt(fan_in)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / (n_embd * block_size) ** 0.5
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01   # chhota -> starting loss kam
b2 = torch.randn(vocab_size, generator=g) * 0

# batch norm ke parameters
bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running = torch.ones((1, n_hidden))

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print('total params:', sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
# training loop (minibatch).... same forward -> backward -> update pattern
max_steps = 20000
batch_size = 32

for i in range(max_steps):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    # forward
    emb = C[Xb]                              # yaha character embedding lookup ho raha hai
    embcat = emb.view(emb.shape[0], -1)      # flatten
    hpreact = embcat @ W1 + b1

    # batch norm
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani) / bnstdi + bnbias
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmeani
        bnstd_running = 0.999 * bnstd_running + 0.001 * bnstdi

    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters:
        p.grad = None
    loss.backward()

    # update.... learning rate ko baad me kam kar dete hai
    lr = 0.1 if i < 10000 else 0.01
    for p in parameters:
        p.data += -lr * p.grad

    if i % 2000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

In [ ]:
# train aur dev pe loss.... yaha batch norm ke running stats use karte hai
@torch.no_grad()
def split_loss(split):
    X, Y = {'train': (Xtr, Ytr), 'dev': (Xdev, Ydev), 'test': (Xte, Yte)}[split]
    emb = C[X]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    print(split, F.cross_entropy(logits, Y).item())


split_loss('train')
split_loss('dev')

# **naam generate karo (mlp se)**

ab dekhte hai bigram wale se kitne behtar naam aate hai.

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(15):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        embcat = emb.view(1, -1)
        hpreact = embcat @ W1 + b1
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        if ix == 0:
            break
        out.append(itos[ix])
    print(''.join(out))

# **week 2 ka summary (mera notes)**

- bigram counts -> sabse simple model, nll around 2.45
- wahi bigram neural net se bhi ban gaya (same loss) -> gradient descent kaam karta hai
- mlp + character embeddings se context use kar paye, loss ~2.1 ke aas paas
- galat weight init se starting loss high aata hai aur tanh dead ho jata hai -> kaiming init se theek
- batch norm ne training smooth ki

next week -> rnn, lstm, attention, transformers....